# Experiment 06 — ITA-Conditioned StyleGAN2-ADA on Clinical Melanoma (MILK10k + PAD-UFES-20)

**Goal:** Fine-tune `M500.pkl` (which was trained on **clinical** melanoma photos) using **100% clinical close-up melanoma images** from **MILK10k + PAD-UFES-20** via the ISIC Archive API (`~508` clean clinical melanoma images, zero dermoscopic mixing), conditioned on **ITA skin-tone bins (`--cond=1`, bins 0–5)**.

**Why this fixes Notebooks 04 & 05:**
- **100% Clinical modality:** Uses `image_type:"clinical: close-up"` only — matches `M500.pkl`'s domain (no dermoscopic mixing, no noisy web-scraped Fitzpatrick17k borders).
- **`--cond=1` (ITA-bin conditioning):** Learns lesion diversity across all ~508 clinical melanomas while conditioning skin tone on ITA bins `0–5`.
- **`--augpipe=bg`:** Geometric-only augmentation (no `hue`/`lumaflip`/`saturation` distortion → prevents magenta artifacts).
- **`--freezed=4`:** Preserves `M500.pkl`'s diverse melanoma structure during transfer learning.

**Requirements:**
- Colab **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`)
- Colab Secrets: `DAGSHUB_USERNAME` and `DAGSHUB_TOKEN` enabled

In [ ]:
# Cell 1 — Verify GPU runtime
import os, sys, subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'No GPU detected! Switch Colab Runtime to T4 GPU.'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — Clone StyleGAN3 & project repo + apply PyTorch 2.x/CUDA 13 patches
SG3_DIR = Path('/content/stylegan3')
if not SG3_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/NVlabs/stylegan3.git', str(SG3_DIR)], check=True)

misc_py = SG3_DIR / 'torch_utils' / 'misc.py'
misc_txt = misc_py.read_text()
if 'super().__init__(dataset)' in misc_txt:
    misc_py.write_text(misc_txt.replace('super().__init__(dataset)', 'super().__init__()'))
    print('Patched torch_utils/misc.py for PyTorch 2.x+')

for op_name in ('bias_act.py', 'upfirdn2d.py'):
    op_py = SG3_DIR / 'torch_utils' / 'ops' / op_name
    op_txt = op_py.read_text()
    if 'def _init():' in op_txt and 'return False # patched' not in op_txt:
        op_txt = op_txt.replace('def _init():', 'def _init():\n    return False # patched')
        op_py.write_text(op_txt)
        print(f'Patched {op_name}')

if str(SG3_DIR) not in sys.path:
    sys.path.insert(0, str(SG3_DIR))

REPO_ROOT = Path('/content/bias-aware-stylegan')
if not REPO_ROOT.exists():
    subprocess.run([
        'git', 'clone', '-b', 'experiment/m500-baseline',
        'https://github.com/Muneeb-Ahmad-404/bias-aware-stylegan.git', str(REPO_ROOT)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', 'origin', 'experiment/m500-baseline'], check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Working dir:', Path.cwd())

In [ ]:
# Cell 3 — Install dependencies & pull M500.pkl via DVC
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'dvc', 'ninja', 'pyspng', 'scipy', 'opencv-python', 'pandas',
    'matplotlib', 'pyyaml', 'click', 'tqdm', 'psutil', 'clean-fid'
], check=True)

from google.colab import userdata
dagshub_user = userdata.get('DAGSHUB_USERNAME')
dagshub_token = userdata.get('DAGSHUB_TOKEN')
assert dagshub_user and dagshub_token, 'Set DAGSHUB_USERNAME and DAGSHUB_TOKEN in Colab Secrets!'

subprocess.run([sys.executable, '-m', 'dvc', 'remote', 'modify', '--local', 'origin_processed', 'auth', 'basic'], check=True)
subprocess.run([sys.executable, '-m', 'dvc', 'remote', 'modify', '--local', 'origin_processed', 'user', dagshub_user], check=True)
subprocess.run([sys.executable, '-m', 'dvc', 'remote', 'modify', '--local', 'origin_processed', 'password', dagshub_token], check=True)
subprocess.run([sys.executable, '-m', 'dvc', 'pull', 'models/pretrained/M500.pkl.dvc'], check=True)
print('M500.pkl size:', Path('models/pretrained/M500.pkl').stat().st_size // (1024*1024), 'MB')

In [ ]:
# Cell 4 — Download 100% clinical close-up melanoma images (MILK10k + PAD-UFES-20 via ISIC API),
# deduplicate via SHA-256, and label with ITA bins (0..5)
import hashlib, json, shutil, urllib.parse, urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from tqdm import tqdm
from src.evaluation.ita import ITA_BINS, compute_image_ita

RAW_CLIN_DIR = Path('/content/clinical_melanoma_raw')
RAW_CLIN_DIR.mkdir(parents=True, exist_ok=True)

COND_DIR = Path('/content/clinical_ita_conditioned')
if COND_DIR.exists():
    shutil.rmtree(COND_DIR)
COND_DIR.mkdir(parents=True, exist_ok=True)

queries = [
    'image_type:"clinical: close-up" AND diagnosis_3:"Melanoma, NOS"',
    'image_type:"clinical: close-up" AND diagnosis_3:"Melanoma Invasive"',
    'image_type:"clinical: close-up" AND diagnosis_3:"Melanoma in situ"',
]

records = {}
for q in queries:
    url = f'https://api.isic-archive.com/api/v2/images/search/?query={urllib.parse.quote(q)}&limit=100'
    while url:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=30) as r:
            data = json.loads(r.read().decode())
        for item in data.get('results', []):
            isic_id = item['isic_id']
            img_url = item.get('files', {}).get('full', {}).get('url')
            if img_url:
                records[isic_id] = img_url
        url = data.get('next')

print(f'Found {len(records)} unique clinical close-up melanoma IDs on ISIC Archive')

def download_one(pair):
    isic_id, img_url = pair
    dest = RAW_CLIN_DIR / f'{isic_id}.jpg'
    if not dest.exists():
        req = urllib.request.Request(img_url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=30) as r:
            dest.write_bytes(r.read())
    return dest

with ThreadPoolExecutor(max_workers=8) as ex:
    raw_paths = list(tqdm(ex.map(download_one, records.items()), total=len(records), desc='Downloading clinical melanomas'))

# Deduplicate via SHA-256 (FR2), resize to 512x512 PNG, and label with ITA bins (0..5)
bin_to_id = {name: idx for idx, (name, _, _) in enumerate(ITA_BINS)}
seen_hashes = set()
duplicates_removed = 0
labels = []
bin_counts = Counter()

for src in tqdm(sorted(raw_paths), desc='Deduplicating & ITA labelling'):
    digest = hashlib.sha256(src.read_bytes()).hexdigest()
    if digest in seen_hashes:
        duplicates_removed += 1
        continue
    seen_hashes.add(digest)

    ita_res = compute_image_ita(src)
    bin_id = bin_to_id.get(ita_res.skin_tone_group)
    if bin_id is None:
        continue

    out_name = f'{src.stem}.png'
    with Image.open(src) as img:
        rgb = img.convert('RGB')
        if rgb.size != (512, 512):
            rgb = rgb.resize((512, 512), Image.Resampling.LANCZOS)
        rgb.save(COND_DIR / out_name, format='PNG')

    labels.append([out_name, bin_id])
    bin_counts[bin_id] += 1

with open(COND_DIR / 'dataset.json', 'w') as f:
    json.dump({'labels': labels}, f, indent=2)

print(f'\nDuplicates removed: {duplicates_removed}')
print(f'Total clean clinical melanoma images: {len(labels)}')
for idx, (name, _, _) in enumerate(ITA_BINS):
    print(f'  {idx} = {name}: {bin_counts.get(idx, 0)} images')

In [ ]:
# Cell 5 — Pack conditioned clinical dataset into StyleGAN ZIP (preserves dataset.json labels)
import zipfile

DATASET_ZIP = Path('/content/clinical_ita_conditioned.zip')
if DATASET_ZIP.exists():
    DATASET_ZIP.unlink()

subprocess.run([
    sys.executable, str(SG3_DIR / 'dataset_tool.py'),
    f'--source={COND_DIR}',
    f'--dest={DATASET_ZIP}',
    '--resolution=512x512'
], check=True)

with zipfile.ZipFile(DATASET_ZIP) as zf:
    assert 'dataset.json' in zf.namelist(), 'dataset.json missing in ZIP!'
    packed = json.loads(zf.read('dataset.json'))
    print(f'Dataset ZIP ready: {DATASET_ZIP.stat().st_size // (1024*1024)} MB | Labels: {len(packed["labels"])}')

In [ ]:
# Cell 6 — Convert unconditional M500.pkl (c_dim=0) -> conditional (c_dim=6) and fine-tune
# --cond=1       : class conditioning on ITA bins 0..5
# --augpipe=bg    : geometric-only augmentation (no color distortion -> no magenta glitches)
# --freezed=4     : preserve M500's lesion diversity during transfer learning
import copy, pickle, torch, legacy, dnnlib

COND_INIT_PKL = Path('/content/M500_cond6_init.pkl')
if not COND_INIT_PKL.exists():
    with open('models/pretrained/M500.pkl', 'rb') as f:
        src = legacy.load_network_pkl(f)
    dst = {}
    for key in ('G', 'G_ema', 'D'):
        m_src = src[key]
        init_kwargs = copy.deepcopy(m_src.init_kwargs)
        init_kwargs['c_dim'] = 6
        m_dst = type(m_src)(**init_kwargs)
        src_sd = m_src.state_dict()
        dst_sd = m_dst.state_dict()
        for k, v in src_sd.items():
            if k in dst_sd and dst_sd[k].shape == v.shape:
                dst_sd[k] = v.clone()
            elif k == 'mapping.fc0.weight' and k in dst_sd:
                # Keep z-path weights identical to M500, zero-init c-path
                dst_sd[k].zero_()
                dst_sd[k][:, :v.shape[1]] = v.clone()
        m_dst.load_state_dict(dst_sd)
        dst[key] = m_dst
    with open(COND_INIT_PKL, 'wb') as f:
        pickle.dump(dst, f)
    print('Converted M500.pkl (c_dim=0) -> M500_cond6_init.pkl (c_dim=6)')

OUT_DIR = Path('/content/training-runs-cond')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# StyleGAN3 train.py has no --augpipe CLI flag; patch AugmentPipe in train.py to geometric-only (bg)
train_py = SG3_DIR / 'train.py'
train_txt = train_py.read_text()
for color_kw in ('brightness=1', 'contrast=1', 'lumaflip=1', 'hue=1', 'saturation=1'):
    train_txt = train_txt.replace(color_kw, color_kw.replace('=1', '=0'))
train_py.write_text(train_txt)
print('Patched train.py AugmentPipe to geometric-only (bg)')

cmd = [
    sys.executable, str(SG3_DIR / 'train.py'),
    f'--outdir={OUT_DIR}',
    '--cfg=stylegan2',
    f'--data={DATASET_ZIP}',
    '--gpus=1',
    '--batch=4',
    '--gamma=5',
    '--cond=1',
    '--mirror=1',
    '--aug=ada',
    '--target=0.6',
    '--freezed=4',
    f'--resume={COND_INIT_PKL}',
    '--kimg=8',
    '--snap=1',
    '--metrics=none',
    '--workers=2',
]
print('Running:', ' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'train.py exited with code {ret}')

In [ ]:
# Cell 7 — Generate 5,004 balanced synthetic clinical melanoma images (834 per ITA bin)
from pathlib import Path
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
import legacy
from src.evaluation.ita import ITA_BINS

OUT_DIR = Path('/content/training-runs-cond')
ckpts = sorted(OUT_DIR.rglob('network-snapshot-*.pkl'))
assert ckpts, 'No checkpoints found!'
best_ckpt = ckpts[-1]
print('Using checkpoint:', best_ckpt)

device = torch.device('cuda')
with open(best_ckpt, 'rb') as f:
    G_ema = legacy.load_network_pkl(f)['G_ema'].to(device).eval()
for p in G_ema.parameters():
    p.requires_grad_(False)

print(f'Generator c_dim = {G_ema.c_dim}')
N_CLASSES = G_ema.c_dim
N_PER_CLASS = 834
BATCH = 10

GEN_DIR = Path('/content/balanced_synthetic_5k')
GEN_DIR.mkdir(parents=True, exist_ok=True)
bin_names = {idx: name for idx, (name, _, _) in enumerate(ITA_BINS)}

with torch.no_grad():
    for class_id in range(N_CLASSES):
        class_name = bin_names.get(class_id, f'bin{class_id}')
        class_dir = GEN_DIR / f'bin{class_id}_{class_name.split(" ")[0]}'
        class_dir.mkdir(parents=True, exist_ok=True)
        seeds = list(range(class_id * N_PER_CLASS, (class_id + 1) * N_PER_CLASS))

        for i in tqdm(range(0, len(seeds), BATCH), desc=f'Bin {class_id} ({class_name})'):
            batch_seeds = seeds[i:i + BATCH]
            bs = len(batch_seeds)
            zs = torch.stack([
                torch.from_numpy(np.random.RandomState(s).randn(G_ema.z_dim)).float()
                for s in batch_seeds
            ]).to(device)
            c = torch.zeros([bs, N_CLASSES], device=device)
            c[:, class_id] = 1.0
            imgs = G_ema(zs, c, truncation_psi=0.85, noise_mode='const')
            imgs = (imgs.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8).cpu().numpy()
            for s, arr in zip(batch_seeds, imgs):
                Image.fromarray(arr, 'RGB').save(class_dir / f'seed_{s:05d}.png')

total_gen = sum(1 for _ in GEN_DIR.rglob('*.png'))
print(f'\nTotal generated: {total_gen} images across {N_CLASSES} ITA bins')

In [ ]:
# Cell 8 — ITA analysis on generated images + comparison chart
import json, shutil
import matplotlib.pyplot as plt
from src.evaluation.ita import ITA_BINS, analyze_image_folder, save_ita_report

FLAT_DIR = Path('/content/balanced_flat')
if FLAT_DIR.exists():
    shutil.rmtree(FLAT_DIR)
FLAT_DIR.mkdir()
for p in GEN_DIR.rglob('*.png'):
    shutil.copy2(p, FLAT_DIR / f'{p.parent.name}_{p.name}')

df_cond, summary_cond = analyze_image_folder(FLAT_DIR)
save_ita_report(df_cond, summary_cond, 'reports/finetune_conditioned_v1', prefix='ita')
print(json.dumps(summary_cond, indent=2))

order = [b[0] for b in ITA_BINS]
baseline_pcts = [37.4, 24.6, 15.0, 12.8, 7.0, 3.2]
cond_pcts = [summary_cond['distribution'][g]['percentage'] for g in order]

x = range(len(order))
width = 0.36
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar([i - width/2 for i in x], baseline_pcts, width, label='M500 Baseline', color='#B4C7E7')
ax.bar([i + width/2 for i in x], cond_pcts, width, label='NB06: Clinical ITA-Conditioned', color='#2E8B57')
ax.axhline(y=100/6, color='gray', linestyle='--', alpha=0.5, label='Perfect balance (16.7%)')
ax.set_xticks(list(x))
ax.set_xticklabels(order, rotation=20)
ax.set_ylabel('Percentage (%)')
ax.set_title('Skin-Tone Distribution: M500 Baseline vs Clinical ITA-Conditioned (N≈5000)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('reports/finetune_conditioned_v1/baseline_vs_conditioned.png', dpi=150)
plt.show()

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for class_id in range(min(N_CLASSES, 6)):
    class_name = bin_names.get(class_id, f'bin{class_id}')
    class_dir = GEN_DIR / f'bin{class_id}_{class_name.split(" ")[0]}'
    samples = sorted(class_dir.glob('*.png'))[:2]
    for row, sp in enumerate(samples):
        axes[row, class_id].imshow(Image.open(sp))
        if row == 0:
            axes[row, class_id].set_title(class_name, fontsize=7)
        axes[row, class_id].axis('off')
plt.suptitle('Generated Clinical Melanoma Samples per ITA Bin (2 per bin)', fontsize=10)
plt.tight_layout()
plt.savefig('reports/finetune_conditioned_v1/per_bin_samples.png', dpi=150)
plt.show()

In [ ]:
# Cell 9 — Compute FID (target: < 50)
from cleanfid import fid

fid_score = fid.compute_fid(str(FLAT_DIR), str(COND_DIR), mode='clean', num_workers=2)
print(f'\n=== FID Score: {fid_score:.2f} (target: < 50) ===')

fid_report = {
    'fid_score': round(float(fid_score), 2),
    'target': 50,
    'pass': bool(fid_score < 50),
    'n_real': len(list(COND_DIR.glob('*.png'))),
    'n_generated': len(list(FLAT_DIR.glob('*.png'))),
}
with open('reports/finetune_conditioned_v1/fid_report.json', 'w') as f:
    json.dump(fid_report, f, indent=2)
print(json.dumps(fid_report, indent=2))

In [ ]:
# Cell 10 — Save checkpoint & reports to Google Drive (optional)
from google.colab import drive
import shutil

drive.mount('/content/drive')
drive_dest = Path('/content/drive/MyDrive/bias-aware-stylegan')
drive_dest.mkdir(parents=True, exist_ok=True)

best_ckpt = sorted(OUT_DIR.rglob('network-snapshot-*.pkl'))[-1]
shutil.copy(best_ckpt, drive_dest / 'M500_cond_clinical_8kimg.pkl')
shutil.copytree('reports/finetune_conditioned_v1', drive_dest / 'reports_conditioned_v1', dirs_exist_ok=True)
print('Saved checkpoint & reports to Google Drive!')